# CineSearch 🎬
Sabrina Aezaz

## Introduction

I watch movies and shows across several streaming services and log everything on Letterboxd, an app film lovers use to track what they've seen and how they rated it. Even so, I often spend 30-60 minutes just deciding what to watch, looking for something I haven't seen that fits my mood. It's a small but recurring friction point, and most people who watch a lot of content probably recognize it.

To address this, I built a movie recommendation app around a simple button-based interface. For each movie, the user taps either **Seen** or **Not seen yet**:

- **Seen:** the user rates the movie and saves the rating.
- **Not seen yet:** the app moves straight to the next movie.

Every choice feeds back into future recommendations. Users can also search for specific movies with the search bar, or describe what they want in plain language using the AI prompt (for example, "something funny and short"). Any movie found through search or the prompt can be added to a **want to watch** list. Users can also skip both and just go through the recommendations one by one.

This notebook builds and tests the recommendation engine behind the app, using the MovieLens dataset.

## Imports

In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from scipy.stats import chi2_contingency
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Data

In [2]:
movies = pd.read_csv("data/movies.csv")
ratings = pd.read_csv("data/ratings.csv")

In [3]:
print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Unique users:", ratings['userId'].nunique())
print("Unique rated movies:", ratings['movieId'].nunique())

Movies: (87585, 3)
Ratings: (32000204, 4)
Unique users: 200948
Unique rated movies: 84432


In [4]:
MIN_RATINGS_PER_MOVIE = 200
MIN_RATINGS_PER_USER = 50
MAX_USERS = 5000

movie_counts = ratings['movieId'].value_counts()
keep_movies = movie_counts[movie_counts >= MIN_RATINGS_PER_MOVIE].index

user_counts = ratings['userId'].value_counts()
keep_users = user_counts[user_counts >= MIN_RATINGS_PER_USER].index
if len(keep_users) > MAX_USERS:
    keep_users = pd.Series(keep_users).sample(MAX_USERS, random_state=42).values

ratings = ratings[ratings['movieId'].isin(keep_movies) & ratings['userId'].isin(keep_users)].copy()
movies = movies[movies['movieId'].isin(keep_movies)].reset_index(drop=True)

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)
print("Users:", ratings['userId'].nunique())


Movies: (9322, 3)
Ratings: (1139587, 4)
Users: 5000


## User Feedback

In MovieLens, every rating is a movie the user has **seen** and rated. A rating of 4 or higher counts as a movie they **liked**.

The dataset has no "Not seen yet" clicks, so the tests below only use "Seen + liked" signals.

In [5]:
feedback = ratings.sort_values('timestamp').copy()
feedback['liked'] = feedback['rating'] >= 4
feedback = feedback[['userId', 'movieId', 'liked']]

liked_only = feedback[feedback['liked']]

# Hide each user's most recent liked movie for testing.
# The models below only learn from the remaining ("train") data.
held_out = liked_only.groupby('userId').tail(1)
train = liked_only.drop(held_out.index)

liked_by_user = train.groupby('userId')['movieId'].apply(list).to_dict()
held_out_by_user = held_out.set_index('userId')['movieId'].to_dict()

print("Users with liked movies:", len(liked_by_user))

Users with liked movies: 4998


## Recommendation Engine
Two approaches:
1. **Genre-based:** recommend movies with genres similar to what the user liked.
2. **Collaborative filtering:** recommend movies that other users who liked the same movies also liked.

In [6]:
# Genre-based similarity
movies['genres_clean'] = movies['genres'].str.replace('|', ' ')
cv = CountVectorizer()
genre_matrix = cv.fit_transform(movies['genres_clean'])
similarity = cosine_similarity(genre_matrix)

In [7]:
# Collaborative filtering similarity (built from training data only)
user_codes = train['userId'].astype('category')
movie_codes = train['movieId'].astype('category')

user_item_sparse = csr_matrix(
    (np.ones(len(train)), (user_codes.cat.codes, movie_codes.cat.codes))
)

item_similarity = cosine_similarity(user_item_sparse.T)
movie_ids_order = movie_codes.cat.categories.tolist()

In [8]:
movie_id_to_idx = {mid: i for i, mid in enumerate(movies['movieId'])}

def recommend_genre(seen_ids, k=10):
    idxs = [movie_id_to_idx[m] for m in seen_ids if m in movie_id_to_idx]
    if not idxs:
        return []
    scores = pd.Series(similarity[idxs].mean(axis=0), index=movies['movieId'])
    scores = scores.drop(labels=seen_ids, errors='ignore')
    return scores.sort_values(ascending=False).head(k).index.tolist()

movieid_to_pos = {mid: i for i, mid in enumerate(movie_ids_order)}

def recommend_cf(seen_ids, k=10):
    idxs = [movieid_to_pos[m] for m in seen_ids if m in movieid_to_pos]
    if not idxs:
        return []
    scores = pd.Series(item_similarity[idxs].mean(axis=0), index=movie_ids_order)
    scores = scores.drop(labels=seen_ids, errors='ignore')
    return scores.sort_values(ascending=False).head(k).index.tolist()

## AI Search
Both let the user find movies and add them to a **want to watch** list.

- **Search bar:** finds movies by title.
- **AI prompt:** a simple keyword parser. It picks up genres the user wants (e.g. "comedy") and genres to avoid (e.g. "no romance"), then filters the user's personalized recommendations.

In [9]:
want_to_watch = []

def add_to_want_to_watch(movie_id):
    if movie_id not in want_to_watch:
        want_to_watch.append(movie_id)

def search_movies(query, k=5):
    found = movies[movies['title'].str.contains(query, case=False, regex=False)]
    return found[['movieId', 'title', 'genres']].head(k)

In [10]:
genre_list = ['comedy', 'romance', 'action', 'horror', 'drama']

def parse_prompt(prompt):
    prompt = prompt.lower()
    include = [g for g in genre_list if g in prompt and f"no {g}" not in prompt]
    exclude = [g for g in genre_list if f"no {g}" in prompt]
    return include, exclude

movie_genres = movies.set_index('movieId')['genres'].str.lower()

def recommend_from_prompt(prompt, seen_ids, k=10):
    include, exclude = parse_prompt(prompt)
    recs = recommend_cf(seen_ids, k=500)
    keep = [m for m in recs
            if all(g in movie_genres[m] for g in include)
            and not any(g in movie_genres[m] for g in exclude)]
    return keep[:k]

## A/B Testing

For each user, we hide their most recent liked movie, give the model the rest, and check whether the hidden movie shows up in the top 10 recommendations (hit rate). Then we compare the two approaches with a chi-square test.

In [11]:
def get_hits(model_recs_func, k=10):
    hits = []
    for user, seen in liked_by_user.items():
        recs = model_recs_func(seen, k)
        hits.append(held_out_by_user[user] in recs)
    return hits

hits_genre = get_hits(recommend_genre)
hits_cf = get_hits(recommend_cf)

print(f"Genre-based hit rate: {np.mean(hits_genre):.1%}  (n={len(hits_genre)})")
print(f"Collaborative filtering hit rate: {np.mean(hits_cf):.1%}  (n={len(hits_cf)})")

table = [
    [sum(hits_genre), len(hits_genre) - sum(hits_genre)],
    [sum(hits_cf), len(hits_cf) - sum(hits_cf)]
]
chi2, p, dof, expected = chi2_contingency(table)
print(f"p-value: {p:.4f}")

Genre-based hit rate: 0.4%  (n=4998)
Collaborative filtering hit rate: 5.9%  (n=4998)
p-value: 0.0000


## Conclusion

- **Result:** collaborative filtering had a top-10 hit rate of **5.9%** vs **0.4%** for genre-based recommendations (n = 4,998 users, p < 0.001).
- **What it means for the app:** recommendations should come from what similar users liked, not just matching genres. Genres alone are too broad to pick out a specific movie someone will like.
- **Limitations:** the data has no "Not seen yet" clicks, so the test only uses seen-and-liked movies. The AI prompt is a simple keyword parser (five genres). The test is offline, using past ratings, not real users.
- **Next steps:** run a live A/B test in the app, measure how often users add recommendations to their want to watch list, and replace the keyword parser with a language model.